# Sistema RAG para análisis de mercado de Ethereum

Pipeline:
1. **Setup**: librerías, dispositivo, modelo de embeddings.
2. **Datos cripto**: carga del CSV histórico con precios y métricas.
3. **Noticias**: descarga del dataset Kaggle (~31k noticias 2021-2023).
4. **Indexado vectorial**: ChromaDB persistente con los 31k docs.
5. **Funciones de inferencia**: `search()`, `responder_pregunta()`.
6. **Evaluación sistemática**: 20 preguntas en 4 bloques.

Modelo LLM: **Gemini 2.5 Flash** (vía API). Free tier: 20 req/día.
Embeddings: **all-MiniLM-L6-v2** (384 dim, ligero, multilingüe básico).
Vector DB: **ChromaDB** persistente local.

## 1. Setup

In [2]:
import os
import ast
import time
import json
import hashlib
import shutil
from datetime import datetime

import torch
import pandas as pd
import chromadb
import kagglehub
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from google import genai

load_dotenv()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}")

# Rutas centralizadas
BASE_DIR     = r"C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF"
RUTA_CSV     = os.path.join(BASE_DIR, "data", "csv", "df_merged.csv")
RUTA_NOTICIAS_DIR = os.path.join(BASE_DIR, "data", "news")
RUTA_NOTICIAS_CSV = os.path.join(RUTA_NOTICIAS_DIR, "cryptonews.csv")
RUTA_TXT     = os.path.join(BASE_DIR, "data", "txt", "contexto_estatico_eth_v2.txt")
RUTA_CHROMA  = os.path.join(BASE_DIR, "chroma_db")
RUTA_RESULTADOS = os.path.join(BASE_DIR, "data", "resultados_eval.json")

# Hiperparámetros del RAG
TOP_K_RETRIEVAL = 30   # cuántos docs recupera la búsqueda inicial
TOP_K_FINAL     = 5    # cuántos se inyectan al prompt final

# Modelo Gemini
MODELO_LLM = "gemini-2.5-flash"
PAUSA_RATE_LIMIT = 13  # segundos entre llamadas (free tier ~5 RPM)

print("Setup OK ✓")

c:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\env_TFG_DEF\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo: cuda
Setup OK ✓


In [3]:
from google import genai
from google.genai import types

client_gemini = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT"):
    """
    task_type: 
      - 'RETRIEVAL_DOCUMENT' para indexar noticias
      - 'RETRIEVAL_QUERY' para búsquedas
    """
    if isinstance(texts, str):
        texts = [texts]
    
    result = client_gemini.models.embed_content(
        model="gemini-embedding-001",
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return [e.values for e in result.embeddings]

# Test
test = embed_texts(["prueba"], task_type="RETRIEVAL_QUERY")
print(f"Dimensiones: {len(test[0])}")  # 3072

Dimensiones: 3072


## 2. Datos cripto históricos

Carga del CSV con precios diarios de BTC/ETH, dominancias, fear&greed,
inflación y fed_rate. Más adelante este DataFrame se enriquecerá con
datos macro (DXY, Nasdaq, oro, US10Y, stablecoins).

In [4]:
df = pd.read_csv(RUTA_CSV, parse_dates=["date"], index_col="date")
df.sort_index(inplace=True)

print(f"Filas: {len(df)}  |  Rango: {df.index.min().date()} → {df.index.max().date()}")
print(f"Columnas ({len(df.columns)}): {df.columns.tolist()}")
df.head(3)

Filas: 2885  |  Rango: 2018-02-01 → 2025-12-25
Columnas (19): ['btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'eth_open', 'eth_high', 'eth_low', 'eth_close', 'eth_volume', 'btc_dominance', 'eth_dominance', 'alt_dominance', 'fear_greed', 'FearGreed_Label', 'inflation', 'btc_mcap', 'eth_mcap', 'fed_rate']


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2018-02-01,10237.299805,10288.799805,8812.280273,9170.540039,9959400448,1119.369995,1161.349976,984.818970,1036.790039,5261680128,0.334251,0.211308,0.454441,30.0,Fear,2.263469,1.703042e+11,1.076635e+11,1.42
2018-02-02,9142.280273,9142.280273,7796.490234,8830.750000,12726899712,1035.770020,1035.770020,757.979980,915.784973,6713290240,0.339622,0.221066,0.439312,15.0,Extreme Fear,2.263469,1.527442e+11,9.942376e+10,1.42
2018-02-03,8852.120117,9430.750000,8251.629883,9174.910156,7263790080,919.210999,991.942993,847.690002,964.018982,3243480064,0.350252,0.209758,0.439990,40.0,Fear,2.263469,1.487152e+11,8.906186e+10,1.42


In [5]:
import yfinance as yf

TICKERS_MACRO = {
    "dxy":     "DX-Y.NYB",   # Indice dolar
    "oro":     "GC=F",       # Oro futuros
    "nasdaq":  "^NDX",       # Nasdaq 100
    "us10y":   "^TNX",       # Yield bono 10 anos USA
    "usdjpy":  "JPY=X",      # USD/JPY
}

def descargar_macro(tickers, start_date, end_date):
    """
    Descarga precios de cierre de los tickers macro y los devuelve
    como un DataFrame con una columna por ticker.
    """
    dfs = {}
    for nombre, ticker in tickers.items():
        print(f"  Descargando {nombre} ({ticker})...")
        try:
            data = yf.download(
                ticker, start=start_date, end=end_date,
                progress=False, auto_adjust=True
            )
            if data.empty:
                print(f"    ⚠️ Sin datos para {ticker}")
                continue
            dfs[nombre] = data["Close"]
        except Exception as e:
            print(f"    ❌ Error en {ticker}: {e}")
    
    df_macro = pd.concat(dfs, axis=1)
    df_macro.columns = list(dfs.keys())
    df_macro.index = pd.to_datetime(df_macro.index).tz_localize(None)
    return df_macro

# Descargar desde la primera fecha de tu CSV cripto
fecha_inicio = df.index.min().strftime("%Y-%m-%d")
fecha_fin    = (df.index.max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

print(f"Descargando datos macro desde {fecha_inicio} hasta {fecha_fin}...")
df_macro = descargar_macro(TICKERS_MACRO, fecha_inicio, fecha_fin)
print(f"\nMacro descargado: {df_macro.shape}")
print(df_macro.tail(3))



Descargando datos macro desde 2018-02-01 hasta 2025-12-26...
  Descargando dxy (DX-Y.NYB)...
  Descargando oro (GC=F)...
  Descargando nasdaq (^NDX)...
  Descargando us10y (^TNX)...
  Descargando usdjpy (JPY=X)...

Macro descargado: (2058, 5)
                  dxy          oro        nasdaq  us10y      usdjpy
Date                                                               
2025-12-22  98.290001  4444.600098  25461.699219  4.169  157.675995
2025-12-23  97.940002  4482.799805  25587.830078  4.169  156.876999
2025-12-24  97.980003  4480.600098  25656.150391  4.136  156.175995


In [6]:
# Merge con forward fill (datos macro no son 24/7, cripto sí)
df_completo = df.join(df_macro, how="left").ffill()

# Variables derivadas macro
df_completo["dxy_chg_30d"]    = df_completo["dxy"].pct_change(30) * 100
df_completo["nasdaq_chg_30d"] = df_completo["nasdaq"].pct_change(30) * 100
df_completo["oro_chg_30d"]    = df_completo["oro"].pct_change(30) * 100
df_completo["us10y_chg_30d"]  = df_completo["us10y"].diff(30)

# Correlaciones rolling 30 dias entre BTC y activos macro
df_completo["btc_ret"]    = df_completo["btc_close"].pct_change()
df_completo["dxy_ret"]    = df_completo["dxy"].pct_change()
df_completo["nasdaq_ret"] = df_completo["nasdaq"].pct_change()
df_completo["oro_ret"]    = df_completo["oro"].pct_change()

df_completo["corr_btc_dxy_30d"]    = df_completo["btc_ret"].rolling(30).corr(df_completo["dxy_ret"])
df_completo["corr_btc_nasdaq_30d"] = df_completo["btc_ret"].rolling(30).corr(df_completo["nasdaq_ret"])
df_completo["corr_btc_oro_30d"]    = df_completo["btc_ret"].rolling(30).corr(df_completo["oro_ret"])

print(f"DataFrame completo: {df_completo.shape}")
print(f"Columnas nuevas:")
nuevas = [c for c in df_completo.columns if c not in df.columns]
for c in nuevas:
    print(f"  - {c}")


DataFrame completo: (2885, 35)
Columnas nuevas:
  - dxy
  - oro
  - nasdaq
  - us10y
  - usdjpy
  - dxy_chg_30d
  - nasdaq_chg_30d
  - oro_chg_30d
  - us10y_chg_30d
  - btc_ret
  - dxy_ret
  - nasdaq_ret
  - oro_ret
  - corr_btc_dxy_30d
  - corr_btc_nasdaq_30d
  - corr_btc_oro_30d


In [7]:
def calcular_snapshot_completo(df, fecha=None):
    """
    Snapshot ampliado: cripto + macro + variables derivadas.
    Devuelve dict con valores listos para inyectar al prompt en lenguaje natural.
    """
    if fecha is None:
        row = df.iloc[-1]
        fecha = df.index[-1]
    else:
        row = df.loc[fecha]
    
    fecha_ts = pd.Timestamp(fecha)
    
    # Drawdown desde ATH
    df_pasado = df.loc[:fecha]
    eth_ath = df_pasado["eth_close"].max()
    btc_ath = df_pasado["btc_close"].max()
    eth_dd  = (row["eth_close"] / eth_ath) - 1
    btc_dd  = (row["btc_close"] / btc_ath) - 1
    
    # Dias post-halving
    halvings = [pd.Timestamp("2012-11-28"), pd.Timestamp("2016-07-09"),
                pd.Timestamp("2020-05-11"), pd.Timestamp("2024-04-19")]
    halvings_pasados = [h for h in halvings if h <= fecha_ts]
    dias_post_halving = (fecha_ts - halvings_pasados[-1]).days if halvings_pasados else None
    
    # Racha de miedo extremo (dias consecutivos con fear_greed < 25)
    fg_serie = df.loc[:fecha, "fear_greed"]
    fg_bajo = (fg_serie < 25).astype(int)
    # Contar racha actual
    racha_miedo = 0
    for v in fg_bajo.iloc[::-1]:
        if v == 1:
            racha_miedo += 1
        else:
            break
    
    return {
        "fecha":                 str(fecha_ts.date()),
        # Cripto
        "precio_eth_usd":        round(row["eth_close"], 2),
        "precio_btc_usd":        round(row["btc_close"], 2),
        "ratio_eth_btc":         round(row["eth_close"] / row["btc_close"], 5),
        "drawdown_eth_pct":      round(eth_dd * 100, 1),
        "drawdown_btc_pct":      round(btc_dd * 100, 1),
        "dominancia_btc_pct":    round(row["btc_dominance"] * 100, 2),
        "dominancia_eth_pct":    round(row["eth_dominance"] * 100, 2),
        "miedo_codicia":         int(row["fear_greed"]),
        "miedo_codicia_etiqueta": row.get("FearGreed_Label", ""),
        "dias_consecutivos_miedo_extremo": racha_miedo,
        "dias_desde_ultimo_halving": dias_post_halving,
        # Macro
        "indice_dolar_dxy":      round(row.get("dxy", 0), 2) if pd.notna(row.get("dxy")) else None,
        "dxy_cambio_30d_pct":    round(row.get("dxy_chg_30d", 0), 2) if pd.notna(row.get("dxy_chg_30d")) else None,
        "oro_usd":               round(row.get("oro", 0), 2) if pd.notna(row.get("oro")) else None,
        "oro_cambio_30d_pct":    round(row.get("oro_chg_30d", 0), 2) if pd.notna(row.get("oro_chg_30d")) else None,
        "nasdaq_100":            round(row.get("nasdaq", 0), 2) if pd.notna(row.get("nasdaq")) else None,
        "nasdaq_cambio_30d_pct": round(row.get("nasdaq_chg_30d", 0), 2) if pd.notna(row.get("nasdaq_chg_30d")) else None,
        "yield_bono_10y_pct":    round(row.get("us10y", 0), 2) if pd.notna(row.get("us10y")) else None,
        "usd_jpy":               round(row.get("usdjpy", 0), 2) if pd.notna(row.get("usdjpy")) else None,
        "tipos_fed_pct":         round(row["fed_rate"], 2),
        "inflacion_pct":         round(row["inflation"], 2),
        # Correlaciones rolling
        "corr_btc_dxy_30d":      round(row.get("corr_btc_dxy_30d", 0), 2) if pd.notna(row.get("corr_btc_dxy_30d")) else None,
        "corr_btc_nasdaq_30d":   round(row.get("corr_btc_nasdaq_30d", 0), 2) if pd.notna(row.get("corr_btc_nasdaq_30d")) else None,
        "corr_btc_oro_30d":      round(row.get("corr_btc_oro_30d", 0), 2) if pd.notna(row.get("corr_btc_oro_30d")) else None,
    }

# Test
snap = calcular_snapshot_completo(df_completo)
print("Snapshot completo del último día disponible:")
for k, v in snap.items():
    print(f"  {k:38s}: {v}")



Snapshot completo del último día disponible:
  fecha                                 : 2025-12-25
  precio_eth_usd                        : 2903.58
  precio_btc_usd                        : 87234.74
  ratio_eth_btc                         : 0.03328
  drawdown_eth_pct                      : -39.9
  drawdown_btc_pct                      : -30.1
  dominancia_btc_pct                    : 57.46
  dominancia_eth_pct                    : 11.68
  miedo_codicia                         : 23
  miedo_codicia_etiqueta                : Extreme Fear
  dias_consecutivos_miedo_extremo       : 3
  dias_desde_ultimo_halving             : 615
  indice_dolar_dxy                      : 97.98
  dxy_cambio_30d_pct                    : -1.69
  oro_usd                               : 4480.6
  oro_cambio_30d_pct                    : 8.25
  nasdaq_100                            : 25656.15
  nasdaq_cambio_30d_pct                 : 2.55
  yield_bono_10y_pct                    : 4.14
  usd_jpy                       

## 3. Noticias históricas (dataset Kaggle)

Dataset `oliviervha/crypto-news` con ~31.000 artículos crypto entre 2021-2023.
Cada artículo trae: fecha, fuente, asunto, texto, título, URL y sentimiento
ya parseado (positive/neutral/negative).

Solo se descarga la primera vez, luego se reutiliza el CSV local.

In [8]:
# Descargar solo si no existe ya el CSV
if not os.path.exists(RUTA_NOTICIAS_CSV):
    os.makedirs(RUTA_NOTICIAS_DIR, exist_ok=True)
    print("Descargando dataset de Kaggle (primera vez)...")
    cache_path = kagglehub.dataset_download("oliviervha/crypto-news")
    for f in os.listdir(cache_path):
        shutil.copy(os.path.join(cache_path, f), os.path.join(RUTA_NOTICIAS_DIR, f))
    print(f"Descargado en: {RUTA_NOTICIAS_DIR}")
else:
    print(f"CSV ya existe: {RUTA_NOTICIAS_CSV}")

CSV ya existe: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\news\cryptonews.csv


In [9]:
# Cargar y procesar el CSV
df_news = pd.read_csv(
    RUTA_NOTICIAS_CSV,
    header=0,
    names=["date", "sentiment", "source", "subject", "text", "title", "url"]
)
df_news = df_news[df_news["date"] != "date"].reset_index(drop=True)

# Parsear sentimiento (viene como string de dict)
def parse_sentiment(s):
    try:
        return ast.literal_eval(s).get("class", "neutral")
    except Exception:
        return "neutral"

df_news["sentiment_class"] = df_news["sentiment"].apply(parse_sentiment)
df_news["date_clean"] = pd.to_datetime(df_news["date"], errors="coerce").dt.strftime("%Y-%m-%d")
df_news = df_news.dropna(subset=["date_clean"]).reset_index(drop=True)

# Asignar categoría: macro / eth / sentiment
PALABRAS_MACRO = ["sec ", "regulation", "ban ", "law ", "government",
                  "federal reserve", "fed ", "sanction", "inflation",
                  "tax ", "policy", "court", "legal", "congress", "senate"]
PALABRAS_ETH   = ["ethereum", " eth ", "bitcoin", " btc ", "defi", "hack",
                  "upgrade", "fork", "mining", "blockchain", "nft", "wallet"]

def asignar_categoria(row):
    texto = f"{row['subject']} {row['title']} {row['text']}".lower()
    if any(w in texto for w in PALABRAS_MACRO):
        return "macro"
    if any(w in texto for w in PALABRAS_ETH):
        return "eth"
    return "sentiment"

df_news["category"] = df_news.apply(asignar_categoria, axis=1)

# Construir lista de docs
def to_docs(df):
    docs = []
    for _, row in df.iterrows():
        titulo = str(row["title"]) if pd.notna(row["title"]) else ""
        texto  = str(row["text"])  if pd.notna(row["text"])  else ""
        combined = f"[{row['sentiment_class'].upper()}] {titulo}. {texto}"[:500]
        if not combined.strip():
            continue
        doc_id = hashlib.md5(combined[:100].encode()).hexdigest()
        docs.append({
            "id":       doc_id,
            "text":     combined,
            "date":     row["date_clean"],
            "source":   str(row["source"]),
            "category": row["category"],
        })
    # Deduplicar por id
    return list({d["id"]: d for d in docs}.values())

docs_historicos = to_docs(df_news)

print(f"Total documentos: {len(docs_historicos)}")
print(f"Rango fechas: {min(d['date'] for d in docs_historicos)} → {max(d['date'] for d in docs_historicos)}")
print("\nDistribución por categoría:")
for cat in ["macro", "eth", "sentiment"]:
    n = sum(1 for d in docs_historicos if d["category"] == cat)
    print(f"  {cat:10s}: {n:>6}")

print("\nEjemplos:")
for d in docs_historicos[:3]:
    print(f"  [{d['date']}] [{d['category']}] {d['text'][:120]}...")

Total documentos: 31007
Rango fechas: 2021-10-12 → 2023-12-19

Distribución por categoría:
  macro     :   4713
  eth       :  20221
  sentiment :   6073

Ejemplos:
  [2023-12-19] [macro] [NEGATIVE] Grayscale CEO Calls for Simultaneous Approval of Spot Products to Level the Field. Grayscale CEO Michael Sonn...
  [2023-12-19] [macro] [NEUTRAL] Indian Government is Actively Collaborating With Crypto Industry: Liminal Custody’s Country Head. In an exclus...
  [2023-12-19] [macro] [POSITIVE] Judge Approves Settlement: Binance to Pay $1.5 Billion to CFTC, CZ to Pay $150 Million Fine. According to the...


## 4. Indexado vectorial (ChromaDB persistente)

Indexa los ~31k docs en ChromaDB. **Solo la primera vez tarda ~10-15 min** con
MiniLM en GPU. ChromaDB es persistente (guarda en disco), por lo que en
ejecuciones siguientes solo carga el índice.

In [10]:
chroma_client = chromadb.PersistentClient(path=RUTA_CHROMA)
collection = chroma_client.get_or_create_collection(
    name="crypto_news",
    metadata={"hnsw:space": "cosine"}
)
print(f"ChromaDB OK ✓  |  Docs en colección: {collection.count()}")

ChromaDB OK ✓  |  Docs en colección: 0


In [12]:
def add_documents(docs, batch_size=100):
    """
    Indexa documentos en ChromaDB usando embeddings de Gemini.
    Batch size 100 = límite de la API de Gemini por llamada.
    """
    if not docs:
        return
    total = len(docs)
    for inicio in range(0, total, batch_size):
        lote = docs[inicio:inicio + batch_size]
        textos    = [d["text"] for d in lote]
        ids       = [d["id"]   for d in lote]
        metadatos = [{"date": d["date"], "source": d["source"], "category": d["category"]}
                     for d in lote]
        
        # Embeddings vía API Gemini (task_type para indexar)
        try:
            embeddings = embed_texts(textos, task_type="RETRIEVAL_DOCUMENT")
        except Exception as e:
            print(f"  ⚠️ Error en lote {inicio//batch_size + 1}: {e}")
            print(f"  Esperando 30s antes de reintentar...")
            time.sleep(30)
            try:
                embeddings = embed_texts(textos, task_type="RETRIEVAL_DOCUMENT")
            except Exception as e2:
                print(f"  ❌ Lote saltado: {e2}")
                continue
        
        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=textos,
            metadatas=metadatos
        )
        
        procesados = min(inicio + batch_size, total)
        print(f"  Lote {inicio//batch_size + 1}: {procesados}/{total} indexados")
        
        # Pausa para respetar rate limit de embeddings
        time.sleep(0.5)
    
    print(f"✓ Total en BD: {collection.count()}")


def search(query, n_results=TOP_K_RETRIEVAL, date_from=None, date_to=None):
    """
    Recupera los documentos más similares a la query usando embeddings de Gemini.
    date_from / date_to: filtros opcionales "YYYY-MM-DD".
    """
    if collection.count() == 0:
        return []
    
    # Embedding de la query (task_type para búsqueda)
    query_vec = embed_texts([query], task_type="RETRIEVAL_QUERY")
    
    # Construir filtro de fechas
    where = None
    if date_from and date_to:
        where = {"$and": [{"date": {"$gte": date_from}}, {"date": {"$lte": date_to}}]}
    elif date_from:
        where = {"date": {"$gte": date_from}}
    elif date_to:
        where = {"date": {"$lte": date_to}}
    
    results = collection.query(
        query_embeddings=query_vec,
        n_results=min(n_results, collection.count()),
        where=where,
        include=["documents", "metadatas", "distances"]
    )
    
    return [
        {
            "text": text,
            "date": meta["date"],
            "source": meta["source"],
            "category": meta["category"],
            "similarity": round(1 - dist, 4),
        }
        for text, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]

## 5. Funciones de búsqueda y de inferencia

In [13]:
def calcular_snapshot_basico(df, fecha=None):
    """
    Devuelve un dict con las variables del snapshot dinámico para una fecha dada.
    Si fecha es None, usa la última disponible.

    NOTA: versión básica. Más adelante se ampliará con macro (DXY, Nasdaq, etc.)
    y con cálculo de patrones activos.
    """
    if fecha is None:
        row = df.iloc[-1]
        fecha = df.index[-1]
    else:
        row = df.loc[fecha]

    # Drawdown desde ATH (usando solo info hasta esa fecha)
    df_pasado = df.loc[:fecha]
    eth_ath = df_pasado["eth_close"].max()
    btc_ath = df_pasado["btc_close"].max()
    eth_dd  = (row["eth_close"] / eth_ath) - 1
    btc_dd  = (row["btc_close"] / btc_ath) - 1

    # Días desde último halving
    halvings = [pd.Timestamp("2012-11-28"), pd.Timestamp("2016-07-09"),
                pd.Timestamp("2020-05-11"), pd.Timestamp("2024-04-19")]
    halvings_pasados = [h for h in halvings if h <= pd.Timestamp(fecha)]
    dias_post_halving = (pd.Timestamp(fecha) - halvings_pasados[-1]).days if halvings_pasados else None

    # Ratio ETH/BTC
    eth_btc_ratio = row["eth_close"] / row["btc_close"]

    return {
        "fecha": str(pd.Timestamp(fecha).date()),
        "eth_close": round(row["eth_close"], 2),
        "btc_close": round(row["btc_close"], 2),
        "eth_btc_ratio": round(eth_btc_ratio, 5),
        "drawdown_ath_eth": round(eth_dd * 100, 1),
        "drawdown_ath_btc": round(btc_dd * 100, 1),
        "btc_dominance": round(row["btc_dominance"] * 100, 2),
        "eth_dominance": round(row["eth_dominance"] * 100, 2),
        "fear_greed": int(row["fear_greed"]),
        "fear_greed_label": row.get("FearGreed_Label", ""),
        "dias_post_halving": dias_post_halving,
        "fed_rate": round(row["fed_rate"], 2),
        "inflation": round(row["inflation"], 2),
    }

snap = calcular_snapshot_basico(df)
print("Snapshot del último día disponible:")
for k, v in snap.items():
    print(f"  {k:25s}: {v}")

Snapshot del último día disponible:
  fecha                    : 2025-12-25
  eth_close                : 2903.58
  btc_close                : 87234.74
  eth_btc_ratio            : 0.03328
  drawdown_ath_eth         : -39.9
  drawdown_ath_btc         : -30.1
  btc_dominance            : 57.46
  eth_dominance            : 11.68
  fear_greed               : 23
  fear_greed_label         : Extreme Fear
  dias_post_halving        : 615
  fed_rate                 : 3.84
  inflation                : 2.7


In [15]:
# Cargar TXT general (patrones de mercado cripto)
RUTA_TXT_GENERAL = os.path.join(BASE_DIR, "data", "txt", "contexto_estatico_eth_v2.txt")
with open(RUTA_TXT_GENERAL, "r", encoding="utf-8") as f:
    CONTEXTO_ESTATICO = f.read()

# Cargar TXT específico de Ethereum
RUTA_TXT_ETH = os.path.join(BASE_DIR, "data", "txt", "contexto_ethereum_v1.txt")
with open(RUTA_TXT_ETH, "r", encoding="utf-8") as f:
    CONTEXTO_ETHEREUM = f.read()

print(f"TXT general cargado:   {len(CONTEXTO_ESTATICO)} caracteres")
print(f"TXT Ethereum cargado:  {len(CONTEXTO_ETHEREUM)} caracteres")
print(f"Total contexto:        ~{(len(CONTEXTO_ESTATICO) + len(CONTEXTO_ETHEREUM)) // 4} tokens aprox.")

TXT general cargado:   33048 caracteres
TXT Ethereum cargado:  23593 caracteres
Total contexto:        ~14160 tokens aprox.


In [17]:
from google import genai

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
print("Cliente Gemini OK ✓")

# Cargar también el TXT específico de Ethereum (descargado aparte)
RUTA_TXT_ETH = os.path.join(BASE_DIR, "data", "txt", "contexto_ethereum_v1.txt")
with open(RUTA_TXT_ETH, "r", encoding="utf-8") as f:
    CONTEXTO_ETHEREUM = f.read()

print(f"TXT Ethereum cargado: {len(CONTEXTO_ETHEREUM)} caracteres")


def snapshot_a_lenguaje_natural(snap):
    """
    Convierte el dict del snapshot en un texto en castellano natural,
    con nombres descriptivos en vez de variables técnicas.
    """
    lineas = [f"Fecha del análisis: {snap['fecha']}", ""]
    
    # Cripto
    lineas.append("DATOS DEL MERCADO CRIPTO:")
    lineas.append(f"  Precio de Ethereum: {snap['precio_eth_usd']} dólares")
    lineas.append(f"  Precio de Bitcoin: {snap['precio_btc_usd']} dólares")
    lineas.append(f"  Ratio Ethereum/Bitcoin: {snap['ratio_eth_btc']}")
    lineas.append(f"  Caída de Ethereum desde su máximo histórico: {snap['drawdown_eth_pct']}%")
    lineas.append(f"  Caída de Bitcoin desde su máximo histórico: {snap['drawdown_btc_pct']}%")
    lineas.append(f"  Dominancia de Bitcoin en el mercado cripto: {snap['dominancia_btc_pct']}%")
    lineas.append(f"  Dominancia de Ethereum en el mercado cripto: {snap['dominancia_eth_pct']}%")
    lineas.append(f"  Índice de miedo y codicia: {snap['miedo_codicia']} ({snap['miedo_codicia_etiqueta']})")
    lineas.append(f"  Días consecutivos en miedo extremo: {snap['dias_consecutivos_miedo_extremo']}")
    if snap['dias_desde_ultimo_halving'] is not None:
        lineas.append(f"  Días desde el último halving de Bitcoin: {snap['dias_desde_ultimo_halving']}")
    
    # Macro
    lineas.append("")
    lineas.append("CONTEXTO MACRO-FINANCIERO:")
    if snap.get('indice_dolar_dxy'):
        lineas.append(f"  Índice del dólar (DXY): {snap['indice_dolar_dxy']} (cambio 30 días: {snap['dxy_cambio_30d_pct']}%)")
    if snap.get('oro_usd'):
        lineas.append(f"  Oro: {snap['oro_usd']} dólares por onza (cambio 30 días: {snap['oro_cambio_30d_pct']}%)")
    if snap.get('nasdaq_100'):
        lineas.append(f"  Nasdaq 100: {snap['nasdaq_100']} (cambio 30 días: {snap['nasdaq_cambio_30d_pct']}%)")
    if snap.get('yield_bono_10y_pct'):
        lineas.append(f"  Yield del bono americano a 10 años: {snap['yield_bono_10y_pct']}%")
    if snap.get('usd_jpy'):
        lineas.append(f"  Tipo de cambio USD/JPY: {snap['usd_jpy']}")
    lineas.append(f"  Tipos de la Reserva Federal: {snap['tipos_fed_pct']}%")
    lineas.append(f"  Inflación interanual EEUU: {snap['inflacion_pct']}%")
    
    # Correlaciones
    if snap.get('corr_btc_nasdaq_30d') is not None:
        lineas.append("")
        lineas.append("CORRELACIONES RECIENTES (ventana 30 días):")
        lineas.append(f"  Bitcoin vs Nasdaq 100: {snap['corr_btc_nasdaq_30d']}")
        lineas.append(f"  Bitcoin vs índice del dólar: {snap['corr_btc_dxy_30d']}")
        lineas.append(f"  Bitcoin vs oro: {snap['corr_btc_oro_30d']}")
    
    return "\n".join(lineas)


def construir_prompt(pregunta, snapshot=None, noticias=None):
    """
    Construye el prompt completo con lenguaje natural.
    Inyecta TXT general + TXT Ethereum + snapshot + noticias + pregunta.
    """
    # Snapshot en lenguaje natural
    snap_txt = snapshot_a_lenguaje_natural(snapshot) if snapshot else "Sin datos del día disponibles."
    
    # Noticias
    if noticias:
        not_txt = "\n".join(
            f"  [{n['date']}] [{n['category']}] {n['source']}: {n['text']}"
            for n in noticias
        )
    else:
        not_txt = "  Sin noticias recuperadas."
    
    return f"""Eres un analista experto en mercados de criptomonedas, especializado en Ethereum.
Habla siempre en castellano natural. NUNCA uses nombres técnicos de variables 
(como fear_greed, btc_dominance, vol_percentil) en tu respuesta. En su lugar, usa 
descripciones humanas: "el índice de miedo y codicia", "la dominancia de Bitcoin", 
"la volatilidad reciente", etc.

Si te falta información para responder, dilo nombrando el dato en castellano natural.
Por ejemplo: "Falta saber cuántos días lleva el mercado en miedo extremo".

No inventes datos. Cita los patrones por su ID cuando los uses.

<CONTEXTO_GENERAL>
{CONTEXTO_ESTATICO}
</CONTEXTO_GENERAL>

<CONTEXTO_ESPECIFICO_ETHEREUM>
{CONTEXTO_ETHEREUM}
</CONTEXTO_ESPECIFICO_ETHEREUM>

<DATOS_DEL_DIA>
{snap_txt}
</DATOS_DEL_DIA>

<NOTICIAS_RELEVANTES>
{not_txt}
</NOTICIAS_RELEVANTES>

<PREGUNTA>
{pregunta}
</PREGUNTA>
"""

# Ahora la función responder_pregunta() funciona igual pero con el snapshot
# enriquecido y el TXT de Ethereum incluido. Solo hay que reemplazar la
# llamada a calcular_snapshot_basico() por calcular_snapshot_completo().

def responder_pregunta(pregunta, fecha=None, usar_rag=True, top_k=TOP_K_FINAL):
    """Pipeline completo con snapshot enriquecido y TXT Ethereum."""
    snap = calcular_snapshot_completo(df_completo, fecha=fecha)
    noticias = []
    if usar_rag and collection.count() > 0:
        date_to = snap["fecha"]
        noticias = search(pregunta, n_results=top_k, date_to=date_to)
    
    prompt = construir_prompt(pregunta, snapshot=snap, noticias=noticias)
    
    try:
        respuesta = client.models.generate_content(
            model=MODELO_LLM, contents=prompt
        ).text
    except Exception as e:
        respuesta = f"❌ ERROR: {e}"
    
    log = {
        "fecha_analisis": snap["fecha"],
        "pregunta": pregunta,
        "snapshot": snap,
        "noticias_recuperadas": [
            {"date": n["date"], "sim": n["similarity"], "text": n["text"][:120]}
            for n in noticias
        ],
        "respuesta": respuesta,
    }
    return respuesta, log

# Test
respuesta, log = responder_pregunta(
    "¿En qué fase del ciclo está Ethereum según los patrones generales y específicos?"
)
print("="*70)
print("RESPUESTA:")
print(respuesta)


Cliente Gemini OK ✓
TXT Ethereum cargado: 23593 caracteres
RESPUESTA:
Como analista experto en mercados de criptomonedas, y con especialización en Ethereum, te proporciono una lectura del estado actual de la fase del ciclo de Ethereum, integrando los patrones generales y específicos con los datos del día 25 de diciembre de 2025.

Actualmente, Ethereum se encuentra en una fase que puede describirse como de **corrección normal dentro de un ciclo potencialmente maduro, pero con una debilidad estructural persistente y notable frente a Bitcoin y otras alternativas.**

Aquí te desgloso los puntos clave:

**1. Regresión en el Ciclo General de Bitcoin y la Posición de Ethereum:**

*   **Pico del ciclo de Bitcoin ya superado:** Con 615 días desde el último halving de Bitcoin, nos encontramos fuera de la ventana histórica de 400-550 días compatible con un pico de ciclo (P1). De hecho, ya hemos superado esa ventana, lo que sugiere un régimen históricamente compatible con un mercado bajista o una 

## 6. Evaluación sistemática (20 preguntas)

**Atención:** consume cuota de Gemini free tier (20 req/día).
Si llegas al límite, parar y continuar al día siguiente, o pasar a `gemini-2.5-flash-lite`.

In [ ]:
preguntas = [
    # BLOQUE 1 — Hechos históricos verificables
    "¿En qué año ocurrió el halving de Bitcoin de 2020 y cuánto bajó la recompensa de bloque? ¿Qué efecto histórico tuvo en el precio en los 12 meses siguientes?",
    "¿Cuál fue el precio máximo histórico (ATH) de Ethereum en USD y en qué fecha exacta ocurrió?",
    "Durante el ciclo alcista de 2017, ¿cuál fue el rango de precio de Ethereum desde su mínimo hasta su máximo? ¿Cuánto tiempo duró aproximadamente ese movimiento?",
    "¿Cuántos halvings de Bitcoin han ocurrido hasta la fecha y cuáles son sus fechas aproximadas?",
    "¿En qué rango de precios cotizaba Bitcoin durante el 'crypto winter' de 2018-2019?",
    # BLOQUE 2 — Patrones de ciclos y correlaciones
    "Basándote únicamente en el contexto proporcionado, ¿cuál es la relación histórica entre el halving de Bitcoin y el inicio de un bull market en Ethereum? ¿Cuántos meses de desfase suele observarse?",
    "¿Existe evidencia histórica de que Ethereum supere el rendimiento de Bitcoin durante la fase tardía de un ciclo alcista? Cita ciclos concretos para justificar tu respuesta.",
    "¿Qué es el ratio ETH/BTC y qué indica históricamente cuando ese ratio sube? ¿En qué ciclos ha marcado máximos?",
    "Describe las 4 fases clásicas de un ciclo de mercado cripto (acumulación, markup, distribución, markdown) y ejemplifícalas con fechas reales de BTC o ETH.",
    "¿Por qué algunos analistas sugieren que Ethereum opera en ciclos de 8 años mientras Bitcoin opera en ciclos de 4 años? ¿Qué datos históricos apoyan o refutan esa hipótesis?",
    # BLOQUE 3 — Razonamiento con incertidumbre
    "Con la información disponible, ¿puede afirmarse que el ciclo de 4 años de Bitcoin seguirá repitiéndose en el futuro? ¿Qué factores podrían romper ese patrón?",
    "¿Es posible predecir con precisión el precio máximo de Ethereum en un ciclo dado, basándose solo en datos de ciclos anteriores? Explica las limitaciones de ese enfoque.",
    "Si el contexto proporcionado no incluye datos sobre el precio actual de Bitcoin, ¿en qué fase del ciclo podría estar el mercado según indicadores on-chain históricos? Sé explícito sobre qué información te falta para responder con certeza.",
    "¿Qué métricas on-chain históricas han sido más útiles para identificar el techo de un ciclo alcista en Bitcoin? ¿Son igualmente válidas para Ethereum?",
    "¿Cuál es la diferencia estructural entre el bull market de 2017 y el de 2021 en términos de quién participó y cómo afectó eso a la volatilidad y la duración del ciclo?",
    # BLOQUE 4 — Trampas anti-alucinación
    "¿Cuál será el precio de Bitcoin en el próximo halving? Responde solo con información del contexto proporcionado, sin especular.",
    "¿Qué indicadores técnicos específicos señalaron exactamente el techo del ciclo de 2021 en Ethereum el día 10 de noviembre? Si no tienes esa información en el contexto, indícalo explícitamente.",
    "Según el contexto que tienes disponible, ¿existe algún patrón que permita predecir el momento exacto en que termina la fase de distribución de un ciclo? Si no hay evidencia suficiente, di por qué.",
    "¿Puede el comportamiento del precio de Ethereum en ciclos pasados utilizarse como modelo predictivo fiable para el siguiente ciclo? Lista explícitamente los supuestos que tendrías que asumir para que esa predicción sea válida.",
    "¿Qué información adicional necesitarías, más allá del contexto proporcionado, para hacer una predicción fundamentada sobre el próximo techo de ciclo de Bitcoin o Ethereum?",
]

BLOQUES = {
    "BLOQUE 1 — Hechos históricos verificables":  range(0, 5),
    "BLOQUE 2 — Patrones de ciclos y correlaciones": range(5, 10),
    "BLOQUE 3 — Razonamiento con incertidumbre":  range(10, 15),
    "BLOQUE 4 — Trampas anti-alucinación":         range(15, 20),
}

print(f"Eval set: {len(preguntas)} preguntas en {len(BLOQUES)} bloques")

In [ ]:
resultados = []

for i, pregunta in enumerate(preguntas):
    bloque_actual = next(nombre for nombre, rango in BLOQUES.items() if i in rango)

    print(f"\n{'='*70}")
    print(f"[{bloque_actual}]")
    print(f"PREGUNTA {i+1}/{len(preguntas)}: {pregunta}")
    print("="*70)

    respuesta, log = responder_pregunta(pregunta, usar_rag=True)
    print(respuesta)

    resultados.append({
        "bloque":   bloque_actual,
        "numero":   i + 1,
        "pregunta": pregunta,
        "respuesta": respuesta,
        "log":      log,
    })

    if i < len(preguntas) - 1:
        print(f"\n⏳ Esperando {PAUSA_RATE_LIMIT}s para respetar rate limit...")
        time.sleep(PAUSA_RATE_LIMIT)

# Guardar resultados
os.makedirs(os.path.dirname(RUTA_RESULTADOS), exist_ok=True)
with open(RUTA_RESULTADOS, "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print(f"\n✓ Resultados guardados en: {RUTA_RESULTADOS}")
print(f"  Total preguntas evaluadas: {len(resultados)}")
print(f"  Errores: {sum(1 for r in resultados if r['respuesta'].startswith('❌'))}")